# Triton으로 벡터 더하기

출처: [Triton 문서의 벡터 덧셈 예제](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html#sphx-glr-getting-started-tutorials-01-vector-add-py)

위 문서의 예제를 따라 Triton의 기본 커널을 만들어 보겠습니다.

Triton에서는 1차원 그리드의 블록을 실행해 두 벡터를 더할 수 있습니다. 커널을 실행하는 Python 함수(`add`)와 실제 연산을 수행하는 커널(`add_kernel`)을 작성합니다. 각 프로그램 인스턴스는 크기가 `1024`인 1차원 타일을 처리합니다.

먼저 Python 래퍼를 살펴보겠습니다.

1. Triton 함수에 전달할 출력 벡터를 먼저 할당합니다.
2. `grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)`는 1차원 실행 그리드를 생성합니다. 블록 수는 `ceil(n_elements / 1024)`입니다.
3. 그리드를 지정해 `add_kernel`을 호출합니다. 주요 연산은 `add_kernel`이 수행합니다.

In [21]:
import torch

import triton
import triton.language as tl

def add(x, y):
    output = torch.empty_like(x)
    n_elements = output.numel()
    
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    
    return output

이제 커널을 살펴보겠습니다. `triton.jit`은 커널을 컴파일합니다. 커널은 한 번에 하나의 프로그램 인스턴스(블록에 해당하는 단위)에서 실행된다는 점을 기억하세요.

1. 커널은 벡터 데이터 자체가 아니라 각 벡터의 첫 번째 요소를 가리키는 포인터를 받습니다. 필요한 데이터를 읽으려면 이 포인터를 사용해야 합니다.
2. `tl.constexpr`로 블록 크기를 상수로 전달합니다. 이 값으로 읽을 데이터의 양을 지정합니다.
3. `tl.program_id`로 프로그램 인스턴스의 ID를 가져옵니다. 실행 중인 프로그램 수는 `tl.num_programs`로 확인할 수도 있습니다.
4. `block_start`로 이 인스턴스가 처리할 데이터의 시작 위치를 구합니다.
5. [`tl.arange`](https://triton-lang.org/main/python-api/generated/triton.language.arange.html#triton.language.arange)와 오프셋을 사용해 이 인스턴스가 처리할 `1024`개 요소의 인덱스를 지정합니다.
6. 존재하지 않는 요소에 접근하지 않도록 마스크를 사용합니다. `ceil(n_elements / 1024)`로 블록 수를 구했기 때문에 마지막 블록은 `1024`개보다 적은 요소를 처리할 수 있습니다.
7. 필요한 x와 y 데이터를 불러옵니다.
8. 각 스레드에서 더하기를 수행해 결과를 계산합니다.
9. 처리한 데이터를 출력 포인터에 저장합니다.

In [ ]:
@triton.jit
def add_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE # 이 프로그램 인스턴스가 처리할 데이터의 첫 번째 인덱스
    
    offsets = block_start + tl.arange(0, BLOCK_SIZE) # 이 인스턴스가 처리할 모든 데이터 인덱스
    mask = offsets < n_elements # n_elements 이후의 데이터에 접근하지 않도록 제한 (블록 크기 * 블록 수가 n_elements보다 클 수 있음)

    x = tl.load(x_ptr + offsets, mask=mask) # x 데이터 불러오기
    y = tl.load(y_ptr + offsets, mask=mask) # y 데이터 불러오기
    output = x + y # 각 스레드에서 덧셈 수행
    tl.store(output_ptr + offsets, output, mask=mask) # 마스크를 사용해 잘못된 접근을 막으면서 결과를 출력 포인터에 저장

커널은 포인터를 사용해 메모리에서 값을 가져옵니다. 포인터를 첫 번째 요소의 메모리 위치에서 시작하는 1차원 인덱스 벡터라고 생각하면 이해하기 쉽습니다. 입력 텐서는 연속된 메모리에 저장되어 있다고 가정합니다(한 열 이동하면 포인터가 1만큼 증가합니다).

따라서 두 번째 행의 값에 접근하려면 첫 번째 요소의 메모리 위치에 첫 번째 행의 너비를 더합니다. 행 끝 위치는 첫 두 행의 너비만큼 더한 곳입니다. 이를 Triton 코드로 표현하면 `row_2_offset = x_ptr + x_cols + tl.arange(0, x_cols)`입니다.

다음은 값과 이에 대응하는 포인터의 예입니다.

In [4]:
import torch

values = torch.rand((5,2))
pointers = torch.arange(10)

print(values)
print(pointers)

tensor([[0.1970, 0.6839],
        [0.5665, 0.8346],
        [0.8535, 0.1808],
        [0.6675, 0.1602],
        [0.8032, 0.9879]])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


Triton 작업에서 어려운 부분 중 하나는 행과 열 인덱스를 포인터로 변환하는 것입니다. 이후에도 이 작업을 자주 다루게 됩니다.

이제 PyTorch의 벡터 덧셈 결과와 비교해 커널이 올바르게 동작하는지 확인하겠습니다.

In [ ]:
torch.manual_seed(0)
size = 1000
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')
output_torch = x + y
output_triton = add(x, y)
print(
    f'PyTorch와 Triton 결과의 최댓값 차이: '
    f'{torch.max(torch.abs(output_torch - output_triton))}'
)
print(output_torch[:10])
print(output_triton[:10])

The maximum difference between torch and triton is 0.0
tensor([1.3713, 1.3076, 0.4940, 1.2701, 1.2803, 1.1750, 1.1790, 1.4607, 0.3393,
        1.2689], device='cuda:0')
tensor([1.3713, 1.3076, 0.4940, 1.2701, 1.2803, 1.1750, 1.1790, 1.4607, 0.3393,
        1.2689], device='cuda:0')
